In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error

def fit_ols_model(df, feature_cols, target_col):
    # ── Input validation ──────────────────────────────────
    assert all(col in df.columns for col in feature_cols), "feature_cols not all in df"
    assert target_col in df.columns, f"{target_col} not in df"

    # ── Step 1: Extract X and y ───────────────────────────
    X = df[feature_cols].values
    y = df[target_col].values

    # ── Step 2: Standardize features ─────────────────────
    scaler   = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # ── Step 3: Fit model ─────────────────────────────────
    model = LinearRegression()
    model.fit(X_scaled, y)

    # ── Step 4: Predictions + metrics ────────────────────
    y_pred = model.predict(X_scaled)
    r2     = r2_score(y, y_pred)
    mse    = mean_squared_error(y, y_pred)

    # ── Step 5: Coefficients + ranking ───────────────────
    coefficients    = dict(zip(feature_cols, model.coef_))
    feature_ranking = sorted(
        [(name, abs(coef)) for name, coef in coefficients.items()],
        key=lambda x: x[1],
        reverse=True
    )
    top_feature = feature_ranking[0][0]

    return {
        'coefficients'   : coefficients,
        'intercept'      : model.intercept_,
        'r2'             : r2,
        'mse'            : mse,
        'feature_ranking': feature_ranking,
        'top_feature'    : top_feature
    }


# ── Test ──────────────────────────────────────────────────
np.random.seed(42)
n = 500
df = pd.DataFrame({
    'num_quests'      : np.random.randint(1, 20, n),
    'friends_count'   : np.random.randint(0, 100, n),
    'days_since_join' : np.random.randint(1, 365, n),
    'session_length'  : np.random.normal(30, 10, n)
})

result = fit_ols_model(
    df,
    feature_cols=['num_quests', 'friends_count', 'days_since_join'],
    target_col='session_length'
)

assert isinstance(result['coefficients'], dict)
assert set(result['coefficients'].keys()) == {'num_quests', 'friends_count', 'days_since_join'}
assert 0 <= result['r2'] <= 1
assert result['mse'] > 0
assert result['top_feature'] in ['num_quests', 'friends_count', 'days_since_join']
assert result['feature_ranking'][0][0] == result['top_feature']
print("All tests passed ✅")
print(result)